# Process to make the 2nd order consensus

We want to make a second consensus training dataset, by running extractions on the same 1000 images using the models fine-tuned on the original consensus, and then making a new set of consensus transcriptions. This should be a better consensus than the original, because it's made from models with more fine-tuning.


## 1. Model checkpoints to use

We will use the 5 model checkpoints fine-tuned on the original consensus

- Daily_rainfall_sample/outputs/checkpoints/HuggingFaceTB--SmolVLM2-2.2B-Instruct-20260611-095504/HuggingFaceTB--SmolVLM2-2.2B-Instruct
- Daily_rainfall_sample/outputs/checkpoints/ibm-granite--granite-vision-4.1-4b-20260611-123400/ibm-granite--granite-vision-4.1-4b
- Daily_rainfall_sample/outputs/checkpoints/google--gemma-3-4b-it-20260611-121400/google--gemma-3-4b-it
- Daily_rainfall_sample/outputs/checkpoints/google--gemma-4-E4B-it-20260611-102927/google--gemma-4-E4B-it
- Daily_rainfall_sample/outputs/checkpoints/mistralai--Mistral-Small-3.1-24B-Instruct-2503-20260611-103013/mistralai--Mistral-Small-3.1-24B-Instruct-2503

We will use the same set of 1000 images

- documents/DailyRainfall_UK/consensus_1000/images


## 3. Run the extractions

Each command submits one extraction job to Azure. Run them in the terminal. They will complete in about 2.5 hours (+queuing time).

```bash
bash /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/scripts/aml_submit.sh --checkpoint Daily_rainfall_sample/outputs/checkpoints/HuggingFaceTB--SmolVLM2-2.2B-Instruct-20260611-095504/HuggingFaceTB--SmolVLM2-2.2B-Instruct --images-path documents/DailyRainfall_UK/consensus_1000/images --transcriptions-path documents/DailyRainfall_UK/consensus_1000/transcriptions_2 --total-shards 1 --batch-size 50 --extraction-registry outputs/extraction_registry.json extract 
```

```bash
bash /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/scripts/aml_submit.sh --checkpoint Daily_rainfall_sample/outputs/checkpoints/ibm-granite--granite-vision-4.1-4b-20260611-123400/ibm-granite--granite-vision-4.1-4b --images-path documents/DailyRainfall_UK/consensus_1000/images --transcriptions-path documents/DailyRainfall_UK/consensus_1000/transcriptions_2 --total-shards 1 --batch-size 50 --extraction-registry outputs/extraction_registry.json extract 
```

```bash
bash /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/scripts/aml_submit.sh --checkpoint Daily_rainfall_sample/outputs/checkpoints/google--gemma-3-4b-it-20260611-121400/google--gemma-3-4b-it --images-path documents/DailyRainfall_UK/consensus_1000/images --transcriptions-path documents/DailyRainfall_UK/consensus_1000/transcriptions_2 --total-shards 1 --batch-size 50 --extraction-registry outputs/extraction_registry.json extract 
```

```bash
bash /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/scripts/aml_submit.sh --checkpoint Daily_rainfall_sample/outputs/checkpoints/google--gemma-4-E4B-it-20260611-102927/google--gemma-4-E4B-it --images-path documents/DailyRainfall_UK/consensus_1000/images --transcriptions-path documents/DailyRainfall_UK/consensus_1000/transcriptions_2 --total-shards 1 --batch-size 50 --extraction-registry outputs/extraction_registry.json extract 
```

```bash
bash /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/scripts/aml_submit.sh --checkpoint Daily_rainfall_sample/outputs/checkpoints/mistralai--Mistral-Small-3.1-24B-Instruct-2503-20260611-103013/mistralai--Mistral-Small-3.1-24B-Instruct-2503 --images-path documents/DailyRainfall_UK/consensus_1000/images --transcriptions-path documents/DailyRainfall_UK/consensus_1000/transcriptions_2 --total-shards 4 --batch-size 15 --extraction-registry outputs/extraction_registry.json extract 
```


## 4. Download the extractions from Azure

Get the extraction run_names from the extraction registry. In this case, they were:

- SmolVLM: 20260611-151721
- Granite4: 20260611-152900
- Gemma3: 20260611-152933
- Gemma4: 20260611-152958
- Ministral: 20260611-153211

Then download them in turn

```bash
bash scripts/aml_download.sh --run-name 20260611-151721
bash scripts/aml_download.sh --run-name 20260611-152900
bash scripts/aml_download.sh --run-name 20260611-152933
bash scripts/aml_download.sh --run-name 20260611-152958
bash scripts/aml_download.sh --run-name 20260611-153211
```




## 5. Make consensus from the downloaded extractions

Looks for cases where multiple extractions agree and flags those sections as reliable.

```bash
bash scripts/run_consensus_pipeline.sh --dataset-root outputs/consensus_dataset_1000 --variant-name consensus_2nd_order --threshold 3 --null-threshold 5 --precision 3 --extraction-dir outputs/extractions/HuggingFaceTB--SmolVLM2-2.2B-Instruct/20260611-151721 --extraction-dir outputs/extractions/ibm-granite--granite-vision-4.1-4b/20260611-152900 --extraction-dir outputs/extractions/google--gemma-3-4b-it/20260611-152933 --extraction-dir outputs/extractions/google--gemma-4-E4B-it/20260611-152958 --extraction-dir outputs/extractions/mistralai--Mistral-Small-3.1-24B-Instruct-2503/20260611-153211 --validate --validation-sample-denominator 20
```

This will produce a fresh consensus in ```outputs/consensus_dataset_1000/consensus_2nd_order``` with transcriptions (ready to use for further fine-tuning), summary and validation figures for 1/20 of the cases.